In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

# === CONFIGURATION ===
SAMPLE_FOLDER = "All_samples"
DATA_FOLDER = "DATA_week"
MODEL_PATH = "cnn_lstm_model.h5"
SCALER_PATH = "cnn_lstm_scaler.pkl"
WINDOW_SIZE = 120
STEP_SIZE = 80
THRESHOLD = 0.2

# === HELPER FUNCTIONS ===
def parse_filename_time(filename):
    parts = filename.replace(".nc", "").split('_')
    date_str = parts[2]
    start = pd.Timestamp(f"{date_str}T{parts[3]}:{parts[4]}:{parts[5]}")
    end = pd.Timestamp(f"{date_str}T{parts[7]}:{parts[8]}:{parts[9]}")
    return start, end

def extract_features(ds):
    df = pd.DataFrame({
        "Bt": ds["Bt"].values,
        "Density": ds["Density"].values,
        "MLT": ds["MLT"].values,
        "MLat": ds["MLat"].values,
        "Dst": ds["Dst"].values,
        "Kp": ds["Kp"].values,
    })
    epsd = ds["EPSD"].values
    for i in range(epsd.shape[1]):
        df[f"EPSD_f{i}"] = epsd[:, i]
    return df

def get_training_intervals():
    intervals = []
    for f in os.listdir(SAMPLE_FOLDER):
        if f.endswith(".nc") and "2016-03-07_" not in f:
            start, end = parse_filename_time(f)
            intervals.append((start, end))
    return intervals

def get_ground_truth_intervals():
    intervals = []
    for f in os.listdir(SAMPLE_FOLDER):
        if f.endswith(".nc") and "2016-03-07_" in f:
            start, end = parse_filename_time(f)
            intervals.append((start, end))
    return intervals

def label_window(start_time, intervals):
    window_end = start_time + pd.Timedelta(seconds=10)
    for s, e in intervals:
        if s <= window_end and e >= start_time:
            return 1
    return 0

def find_event_boundaries(bt_window):
    grad_bt = np.gradient(bt_window)
    max_grad = np.max(grad_bt)
    min_grad = np.min(grad_bt)

    upper_thresh = 0.2 * max_grad
    lower_thresh = 0.2 * min_grad

    candidates = np.where((grad_bt >= upper_thresh) | (grad_bt <= lower_thresh))[0]

    if len(candidates) >= 2:
        return candidates[0], candidates[-1]
    else:
        return 0, len(bt_window) - 1

# === TRAINING PHASE ===
features_list = []
labels_list = []
intervals = get_training_intervals()

for f in os.listdir(DATA_FOLDER):
    if not f.endswith(".nc"):
        continue
    path = os.path.join(DATA_FOLDER, f)
    ds = xr.open_dataset(path, decode_times=True)
    times = pd.to_datetime(ds["time"].values)
    feats = extract_features(ds).interpolate(limit_direction='both')

    for i in range(0, len(feats) - WINDOW_SIZE, STEP_SIZE):
        window = feats.iloc[i:i+WINDOW_SIZE]
        if window.isnull().values.any():
            continue

        bt_window = window["Bt"].values
        left, right = find_event_boundaries(bt_window)
        window_trimmed = window.iloc[left:right+1]
        t = times[i + left]

        features_list.append(window_trimmed.values)
        labels_list.append(label_window(t, intervals))

# Pad trimmed windows to WINDOW_SIZE if needed
max_len = max(w.shape[0] for w in features_list)
if max_len < WINDOW_SIZE:
    WINDOW_SIZE = max_len  # shrink model input size to max found
features_list = [np.pad(w, ((0, WINDOW_SIZE - len(w)), (0, 0)), mode='edge') for w in features_list]

X = np.array(features_list)
y = np.array(labels_list)

scaler = StandardScaler()
X_reshaped = X.reshape(-1, X.shape[-1])
X_scaled = scaler.fit_transform(X_reshaped).reshape(X.shape)

joblib.dump(scaler, SCALER_PATH)

# Build model
model = Sequential([
    Input(shape=(X.shape[1], X.shape[2])),
    Conv1D(64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    LSTM(64),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train
model.fit(X_scaled, y, epochs=10, batch_size=64, validation_split=0.2,
          callbacks=[EarlyStopping(patience=2)])
model.save(MODEL_PATH)

# === TESTING PHASE ===
test_file = "mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc"
ds_test = xr.open_dataset(os.path.join(DATA_FOLDER, test_file), decode_times=True)
times_test = pd.to_datetime(ds_test["time"].values)
feats_test = extract_features(ds_test).interpolate(limit_direction='both')

X_test, time_indices = [], []
for i in range(0, len(feats_test) - WINDOW_SIZE, STEP_SIZE):
    window = feats_test.iloc[i:i+WINDOW_SIZE]
    if window.isnull().values.any():
        continue
    X_test.append(window.values)
    time_indices.append(times_test[i])

X_test = np.array(X_test)
scaler = joblib.load(SCALER_PATH)
X_test_scaled = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)
model = tf.keras.models.load_model(MODEL_PATH)

probs = model.predict(X_test_scaled).flatten()
pred_intervals = [(time_indices[i], time_indices[i] + pd.Timedelta(seconds=10))
                  for i, p in enumerate(probs) if p >= THRESHOLD]

# === EVALUATION ===
true_intervals = get_ground_truth_intervals()

def overlap(a, b):
    return a[0] <= b[1] and b[0] <= a[1]

def evaluate(preds, truths):
    TP = sum(any(overlap(p, t) for t in truths) for p in preds)
    FP = len(preds) - TP
    FN = sum(not any(overlap(t, p) for p in preds) for t in truths)
    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return precision, recall, f1

precision, recall, f1 = evaluate(pred_intervals, true_intervals)
print("\n\U0001F4C8 Evaluation Metrics:")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model

# === CONFIG ===
MODEL_PATH = "cnn_lstm_model.h5"
SCALER_PATH = "cnn_lstm_scaler.pkl"
WINDOW_SIZE = 120
STEP_SIZE = 40
THRESHOLD = 0.5
OUTPUT_FOLDER = "predicted_events"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def extract_features(ds):
    df = pd.DataFrame({
        "Bt": ds["Bt"].values,
        "Density": ds["Density"].values,
        "MLT": ds["MLT"].values,
        "MLat": ds["MLat"].values,
        "Dst": ds["Dst"].values,
        "Kp": ds["Kp"].values,
    })
    epsd = ds["EPSD"].values
    for i in range(epsd.shape[1]):
        df[f"EPSD_f{i}"] = epsd[:, i]
    return df

def predict_events(ds, features, times, model, scaler, threshold=THRESHOLD):
    events = []
    for i in range(0, len(features) - WINDOW_SIZE, STEP_SIZE):
        window = features.iloc[i:i + WINDOW_SIZE].values
        if np.isnan(window).any():
            continue
        scaled = scaler.transform(window).reshape(1, WINDOW_SIZE, -1)
        prob = model.predict(scaled, verbose=0)[0][0]
        print(f"Window {i}: P(event) = {prob:.2f}")
        if prob >= threshold:
            start_time = times[i]
            end_time = times[i + WINDOW_SIZE - 1]
            events.append((i, start_time, end_time))
    return events

def save_event(ds, i, start_time, end_time, file_base):
    event_ds = ds.isel(time=slice(i, i + WINDOW_SIZE))
    out_name = f"{file_base}_{start_time.strftime('%H_%M_%S')}_to_{end_time.strftime('%H_%M_%S')}.nc"
    out_path = os.path.join(OUTPUT_FOLDER, out_name)
    event_ds.to_netcdf(out_path)
    print(f"✅ Saved event: {out_path}")

def plot_event(ds, i, start_time, end_time):
    bt = ds["Bt"].values
    epsd = ds["EPSD"].values
    freq = ds["frequency"].values
    time = pd.to_datetime(ds["time"].values)

    bt_segment = bt[i:i + WINDOW_SIZE]
    epsd_segment = epsd[i:i + WINDOW_SIZE, :]
    time_segment = time[i:i + WINDOW_SIZE]

    fig, ax1 = plt.subplots(figsize=(12, 6))
    pcm = ax1.pcolormesh(time_segment, freq, epsd_segment.T, shading='auto', cmap='jet')
    ax1.set_yscale('log')
    ax1.set_ylabel("Frequency (Hz)")
    fig.colorbar(pcm, ax=ax1, label='EPSD')

    ax2 = ax1.twinx()
    ax2.plot(time_segment, bt_segment, color='white', linewidth=1.5)
    ax2.set_ylabel("Bt (nT)", color='white')
    ax2.tick_params(axis='y', labelcolor='white')
    ax2.spines['right'].set_color('white')

    ax1.set_title(f"Predicted Event: {start_time} to {end_time}")
    plt.tight_layout()

    jpg_name = f"{start_time.strftime('%Y-%m-%d_%H-%M-%S')}_to_{end_time.strftime('%H-%M-%S')}.jpg"
    jpg_path = os.path.join(OUTPUT_FOLDER, jpg_name)
    plt.savefig(jpg_path, dpi=300)
    plt.close()
    print(f"🖼️ Saved plot: {jpg_path}")

def plot_event(ds, i, start_time, end_time):
    bt = ds["Bt"].values
    epsd = ds["EPSD"].values
    freq = ds["frequency"].values
    time = pd.to_datetime(ds["time"].values)

    bt_segment = bt[i:i + WINDOW_SIZE]
    epsd_segment = epsd[i:i + WINDOW_SIZE, :]
    time_segment = time[i:i + WINDOW_SIZE]

    # === Compute Bt Gradient ===
    bt_grad = np.gradient(bt_segment)
    grad_threshold = 0.5 * np.max(np.abs(bt_grad))
    boundaries = np.where(np.abs(bt_grad) > grad_threshold)[0]

    if len(boundaries) >= 2:
        left, right = boundaries[0], boundaries[-1]
    else:
        # fallback: use full window
        left, right = 0, len(bt_segment) - 1

    width_seconds = (time_segment[right] - time_segment[left]).total_seconds()

    # === Plot EPSD ===
    fig, ax1 = plt.subplots(figsize=(12, 6))
    pcm = ax1.pcolormesh(time_segment, freq, epsd_segment.T, shading='auto', cmap='jet')
    #ax1.set_yscale('log')
    ax1.set_ylabel("Frequency (Hz)")
    fig.colorbar(pcm, ax=ax1, label='EPSD')

    # === Plot Bt ===
    ax2 = ax1.twinx()
    ax2.plot(time_segment, bt_segment, color='white', linewidth=1.5, label='Bt (nT)')
    ax2.axvline(time_segment[left], color='cyan', linestyle='--', label='Event Boundaries')
    ax2.axvline(time_segment[right], color='cyan', linestyle='--')
    ax2.set_ylabel("Bt (nT)", color='white')
    ax2.tick_params(axis='y', labelcolor='white')
    ax2.spines['right'].set_color('white')

    # === Title & Annotation ===
    ax1.set_title(f"Predicted Event: {start_time} to {end_time}\nWidth ≈ {width_seconds:.1f} sec")
    plt.tight_layout()

    # === Save ===
    jpg_name = f"{start_time.strftime('%Y-%m-%d_%H-%M-%S')}_to_{end_time.strftime('%H-%M-%S')}.jpg"
    jpg_path = os.path.join(OUTPUT_FOLDER, jpg_name)
    plt.savefig(jpg_path, dpi=300)
    plt.close()

    print(f"🖼️ Saved plot: {jpg_path} | Width ≈ {width_seconds:.1f} sec")


def run_detection(file_path):
    print(f"\n🚀 Running detection on: {file_path}")
    model = load_model(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)

    ds = xr.open_dataset(file_path, decode_times=True)
    times = pd.to_datetime(ds["time"].values)
    features = extract_features(ds).interpolate(limit_direction='both')

    events = predict_events(ds, features, times, model, scaler)
    print(f"🧠 Detected {len(events)} events.")

    file_base = os.path.basename(file_path).replace(".nc", "")
    for i, start, end in events:
        save_event(ds, i, start, end, file_base)
        plot_event(ds, i, start, end)

# === RUN ===
run_detection("DATA_week/mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc")
#run_detection("DATA_week/mms1_multidimensional_2016-03-01_00_00_00_to_2_59_59.nc")



# Detection based on the Bt gradients (fixed window size)

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

# === CONFIGURATION ===
SAMPLE_FOLDER = "All_samples"
DATA_FOLDER = "DATA_week"
MODEL_PATH = "cnn_lstm_model.h5"
SCALER_PATH = "cnn_lstm_scaler.pkl"
WINDOW_SIZE = 160
STEP_SIZE = 80
THRESHOLD = 0.4

# === HELPER FUNCTIONS ===
def parse_filename_time(filename):
    parts = filename.replace(".nc", "").split('_')
    date_str = parts[2]
    start = pd.Timestamp(f"{date_str}T{parts[3]}:{parts[4]}:{parts[5]}")
    end = pd.Timestamp(f"{date_str}T{parts[7]}:{parts[8]}:{parts[9]}")
    return start, end

def extract_features(ds):
    df = pd.DataFrame({
        "Bt": ds["Bt"].values,
        "Density": ds["Density"].values,
        "MLT": ds["MLT"].values,
        "MLat": ds["MLat"].values,
        "Dst": ds["Dst"].values,
        "Kp": ds["Kp"].values,
    })
    epsd = ds["EPSD"].values
    for i in range(epsd.shape[1]):
        df[f"EPSD_f{i}"] = epsd[:, i]
    return df

def get_training_intervals():
    intervals = []
    for f in os.listdir(SAMPLE_FOLDER):
        if f.endswith(".nc") and "2016-03-07_" not in f:
            start, end = parse_filename_time(f)
            intervals.append((start, end))
    return intervals

def get_ground_truth_intervals():
    intervals = []
    for f in os.listdir(SAMPLE_FOLDER):
        if f.endswith(".nc") and "2016-03-07_" in f:
            start, end = parse_filename_time(f)
            intervals.append((start, end))
    return intervals

def label_window(start_time, intervals):
    window_end = start_time + pd.Timedelta(seconds=10)
    for s, e in intervals:
        if s <= window_end and e >= start_time:
            return 1
    return 0

def detect_bt_boundaries(bt):
    grad_bt = np.gradient(bt)
    max_grad = np.max(grad_bt)
    min_grad = np.min(grad_bt)
    upper_thresh = 0.2 * max_grad
    lower_thresh = 0.2 * min_grad
    candidates = np.where((grad_bt >= upper_thresh) | (grad_bt <= lower_thresh))[0]
    if len(candidates) >= 2:
        return int(candidates[0]), int(candidates[-1])
    return 0, len(bt) - 1

# === TRAINING PHASE ===
features_list = []
labels_list = []
intervals = get_training_intervals()

for f in os.listdir(DATA_FOLDER):
    if not f.endswith(".nc"):
        continue
    path = os.path.join(DATA_FOLDER, f)
    ds = xr.open_dataset(path, decode_times=True)
    times = pd.to_datetime(ds["time"].values)
    feats = extract_features(ds).interpolate(limit_direction='both')

    for i in range(0, len(feats) - WINDOW_SIZE, STEP_SIZE):
        window = feats.iloc[i:i+WINDOW_SIZE]
        if window.isnull().values.any():
            continue

        bt_window = window["Bt"].values
        left, right = detect_bt_boundaries(bt_window)
        window_trimmed = window.iloc[int(left):int(right) + 1]
        t = times[i + int(left)]

        features_list.append(window_trimmed.values)
        labels_list.append(label_window(t, intervals))

# Pad trimmed windows to uniform shape
max_len = max(w.shape[0] for w in features_list)
features_list = [np.pad(w, ((0, max_len - w.shape[0]), (0, 0)), mode='edge') for w in features_list]

X = np.array(features_list)
y = np.array(labels_list)

scaler = StandardScaler()
X_reshaped = X.reshape(-1, X.shape[-1])
X_scaled = scaler.fit_transform(X_reshaped).reshape(X.shape)
joblib.dump(scaler, SCALER_PATH)

# Build and train model
model = Sequential([
    Input(shape=(X.shape[1], X.shape[2])),
    Conv1D(64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    LSTM(64),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_scaled, y, epochs=10, batch_size=64, validation_split=0.2,
          callbacks=[EarlyStopping(patience=2)])
model.save(MODEL_PATH)

# === TESTING PHASE ===
test_file = "mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc"
ds_test = xr.open_dataset(os.path.join(DATA_FOLDER, test_file), decode_times=True)
times_test = pd.to_datetime(ds_test["time"].values)
feats_test = extract_features(ds_test).interpolate(limit_direction='both')

X_test, time_indices = [], []
for i in range(0, len(feats_test) - WINDOW_SIZE, STEP_SIZE):
    window = feats_test.iloc[i:i+WINDOW_SIZE]
    if window.isnull().values.any():
        continue
    X_test.append(window.values)
    time_indices.append(times_test[i])

X_test = np.array(X_test)
scaler = joblib.load(SCALER_PATH)
X_test_scaled = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)
model = tf.keras.models.load_model(MODEL_PATH)

probs = model.predict(X_test_scaled).flatten()
pred_intervals = [(time_indices[i], time_indices[i] + pd.Timedelta(seconds=10))
                  for i, p in enumerate(probs) if p >= THRESHOLD]

true_intervals = get_ground_truth_intervals()

def overlap(a, b):
    return a[0] <= b[1] and b[0] <= a[1]

def evaluate(preds, truths):
    TP = sum(any(overlap(p, t) for t in truths) for p in preds)
    FP = len(preds) - TP
    FN = sum(not any(overlap(t, p) for p in preds) for t in truths)
    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return precision, recall, f1

precision, recall, f1 = evaluate(pred_intervals, true_intervals)
print("\n📈 Evaluation Metrics:")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model

# === CONFIG ===
MODEL_PATH = "cnn_lstm_model.h5"
SCALER_PATH = "cnn_lstm_scaler.pkl"
WINDOW_SIZE = 120
STEP_SIZE = 40
THRESHOLD = 0.5
OUTPUT_FOLDER = "predicted_events_2"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def extract_features(ds):
    df = pd.DataFrame({
        "Bt": ds["Bt"].values,
        "Density": ds["Density"].values,
        "MLT": ds["MLT"].values,
        "MLat": ds["MLat"].values,
        "Dst": ds["Dst"].values,
        "Kp": ds["Kp"].values,
    })
    epsd = ds["EPSD"].values
    for i in range(epsd.shape[1]):
        df[f"EPSD_f{i}"] = epsd[:, i]
    return df

def predict_events(ds, features, times, model, scaler, threshold=THRESHOLD):
    events = []
    for i in range(0, len(features) - WINDOW_SIZE, STEP_SIZE):
        window = features.iloc[i:i + WINDOW_SIZE].values
        if np.isnan(window).any():
            continue
        scaled = scaler.transform(window).reshape(1, WINDOW_SIZE, -1)
        prob = model.predict(scaled, verbose=0)[0][0]
        print(f"Window {i}: P(event) = {prob:.2f}")
        if prob >= threshold:
            start_time = times[i]
            end_time = times[i + WINDOW_SIZE - 1]
            events.append((i, start_time, end_time))
    return events

def save_event(ds, i, start_time, end_time, file_base):
    event_ds = ds.isel(time=slice(i, i + WINDOW_SIZE))
    out_name = f"{file_base}_{start_time.strftime('%H_%M_%S')}_to_{end_time.strftime('%H_%M_%S')}.nc"
    out_path = os.path.join(OUTPUT_FOLDER, out_name)
    event_ds.to_netcdf(out_path)
    print(f"✅ Saved event: {out_path}")

def plot_event(ds, i, start_time, end_time):
    bt = ds["Bt"].values
    epsd = ds["EPSD"].values
    freq = ds["frequency"].values
    time = pd.to_datetime(ds["time"].values)

    bt_segment = bt[i:i + WINDOW_SIZE]
    epsd_segment = epsd[i:i + WINDOW_SIZE, :]
    time_segment = time[i:i + WINDOW_SIZE]

    fig, ax1 = plt.subplots(figsize=(12, 6))
    pcm = ax1.pcolormesh(time_segment, freq, epsd_segment.T, shading='auto', cmap='jet')
    ax1.set_yscale('log')
    ax1.set_ylabel("Frequency (Hz)")
    fig.colorbar(pcm, ax=ax1, label='EPSD')

    ax2 = ax1.twinx()
    ax2.plot(time_segment, bt_segment, color='white', linewidth=1.5)
    ax2.set_ylabel("Bt (nT)", color='white')
    ax2.tick_params(axis='y', labelcolor='white')
    ax2.spines['right'].set_color('white')

    ax1.set_title(f"Predicted Event: {start_time} to {end_time}")
    plt.tight_layout()

    jpg_name = f"{start_time.strftime('%Y-%m-%d_%H-%M-%S')}_to_{end_time.strftime('%H-%M-%S')}.jpg"
    jpg_path = os.path.join(OUTPUT_FOLDER, jpg_name)
    plt.savefig(jpg_path, dpi=300)
    plt.close()
    print(f"🖼️ Saved plot: {jpg_path}")

def plot_event(ds, i, start_time, end_time):
    bt = ds["Bt"].values
    epsd = ds["EPSD"].values
    freq = ds["frequency"].values
    time = pd.to_datetime(ds["time"].values)

    bt_segment = bt[i:i + WINDOW_SIZE]
    epsd_segment = epsd[i:i + WINDOW_SIZE, :]
    time_segment = time[i:i + WINDOW_SIZE]

    # === Compute Bt Gradient ===
    bt_grad = np.gradient(bt_segment)
    grad_threshold = 0.5 * np.max(np.abs(bt_grad))
    boundaries = np.where(np.abs(bt_grad) > grad_threshold)[0]

    if len(boundaries) >= 2:
        left, right = boundaries[0], boundaries[-1]
    else:
        # fallback: use full window
        left, right = 0, len(bt_segment) - 1

    width_seconds = (time_segment[right] - time_segment[left]).total_seconds()

    # === Plot EPSD ===
    fig, ax1 = plt.subplots(figsize=(12, 6))
    pcm = ax1.pcolormesh(time_segment, freq, epsd_segment.T, shading='auto', cmap='jet')
    #ax1.set_yscale('log')
    ax1.set_ylabel("Frequency (Hz)")
    fig.colorbar(pcm, ax=ax1, label='EPSD')

    # === Plot Bt ===
    ax2 = ax1.twinx()
    ax2.plot(time_segment, bt_segment, color='white', linewidth=1.5, label='Bt (nT)')
    ax2.axvline(time_segment[left], color='cyan', linestyle='--', label='Event Boundaries')
    ax2.axvline(time_segment[right], color='cyan', linestyle='--')
    ax2.set_ylabel("Bt (nT)", color='white')
    ax2.tick_params(axis='y', labelcolor='white')
    ax2.spines['right'].set_color('white')

    # === Title & Annotation ===
    ax1.set_title(f"Predicted Event: {start_time} to {end_time}\nWidth ≈ {width_seconds:.1f} sec")
    plt.tight_layout()

    # === Save ===
    jpg_name = f"{start_time.strftime('%Y-%m-%d_%H-%M-%S')}_to_{end_time.strftime('%H-%M-%S')}.jpg"
    jpg_path = os.path.join(OUTPUT_FOLDER, jpg_name)
    plt.savefig(jpg_path, dpi=300)
    plt.close()

    print(f"🖼️ Saved plot: {jpg_path} | Width ≈ {width_seconds:.1f} sec")


def run_detection(file_path):
    print(f"\n🚀 Running detection on: {file_path}")
    model = load_model(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)

    ds = xr.open_dataset(file_path, decode_times=True)
    times = pd.to_datetime(ds["time"].values)
    features = extract_features(ds).interpolate(limit_direction='both')

    events = predict_events(ds, features, times, model, scaler)
    print(f"🧠 Detected {len(events)} events.")

    file_base = os.path.basename(file_path).replace(".nc", "")
    for i, start, end in events:
        save_event(ds, i, start, end, file_base)
        plot_event(ds, i, start, end)

# === RUN ===
run_detection("DATA_week/mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc")
#run_detection("DATA_week/mms1_multidimensional_2016-03-01_00_00_00_to_2_59_59.nc")

